# Task 3: Hypothesis Testing — Risk Drivers Analysis

In this notebook I test whether key risk‑driver features (e.g., Gender, Province) exhibit statistically significant differences in claim frequency, claim severity, or margin.  
Metrics (KPIs):  
- Claim Frequency = proportion of policies with at least one claim.  
- Claim Severity = average claim amount among policies with claims.  
- Margin = TotalPremium − TotalClaims.  

I also check balance of control variables across groups before drawing conclusions.


In [50]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

import statsmodels.api as sm


# adjust as needed
RAW_PATH = "../data/raw/MachineLearningRating_v3.txt"

# attempt to load; may need to adjust delimiter / parsing depending on file format
try:
    df = pd.read_csv(RAW_PATH, sep="|", engine="python")
    print("Loaded as pipe-delimited file")
except Exception as e:
    print("CSV load failed:", e)
    try:
        df = pd.read_csv(RAW_PATH, sep=r"\s+", engine="python")
        print("Loaded as whitespace-delimited")
    except Exception as e2:
        print("Whitespace load failed:", e2)
        df = pd.read_fwf(RAW_PATH)
        print("Loaded via fixed-width")


# Derive KPIs
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]


Loaded as pipe-delimited file


## Helper function & group‑level summary

Define a helper to compute, for each group (e.g. by Province or Gender):

- number of policies  
- number of policies with at least one claim  
- claim frequency (proportion)  
- average claim amount among those with claims  
- average margin  

This will help us compare segments before formal hypothesis testing.


In [51]:
def group_stats(df, group_col):
    grouped = df.groupby(group_col).agg(
        n_policies = ("PolicyID", "count"),
        n_with_claim = ("HasClaim", "sum"),
        freq = ("HasClaim", "mean"),
        mean_claim = ("TotalClaims", lambda x: x[x>0].mean()),
        mean_margin = ("Margin", "mean")
    )
    return grouped

# Example: by Gender
print("=== By Gender ===")
print(group_stats(df, "Gender"))

# Example: by Province
print("=== By Province (top 10 by number of policies) ===")
print(group_stats(df, "Province").sort_values("n_policies", ascending=False).head(10))


=== By Gender ===
               n_policies  n_with_claim      freq    mean_claim  mean_margin
Gender                                                                      
Female               6755            14  0.002073  17874.721303     8.028787
Male                42817            94  0.002195  14858.552294     4.284253
Not specified      940990          2666  0.002833  23530.667678    -3.731550
=== By Province (top 10 by number of policies) ===
               n_policies  n_with_claim      freq    mean_claim  mean_margin
Province                                                                    
Gauteng            393865          1322  0.003356  22243.878396   -13.558894
Western Cape       170796           370  0.002166  28095.849881    -3.414689
KwaZulu-Natal      169781           483  0.002845  29609.487473    -6.433598
North West         143287           349  0.002436  16963.467035    10.958832
Mpumalanga          52718           128  0.002428  15979.553421    15.016059
Eastern

## Test 1: Claim Frequency difference — Gender (Male vs Female)

Null Hypothesis: Claim frequency is the same for male and female policy‑holders.  
Alternative Hypothesis: Claim frequency differs between genders.  
We use a two‑proportion Z‑test.


In [52]:
male = df[df["Gender"] == "Male"]
female = df[df["Gender"] == "Female"]

count = np.array([ male["HasClaim"].sum(), female["HasClaim"].sum() ])
nobs  = np.array([ len(male), len(female) ])

stat, pval = sm.stats.proportions_ztest(count, nobs)
print("Claim Frequency difference (Male vs Female): Z = {:.3f}, p = {:.5f}".format(stat, pval))


Claim Frequency difference (Male vs Female): Z = 0.201, p = 0.84049


### Test 2: Claim Severity difference — Gender (for those with claims)

Null Hypothesis: among policies with a claim, the mean claim amount is the same for male and female.  
Use Welch’s t‑test (unequal variance).

### Test 3: Margin difference — Gender

Null Hypothesis: margin (TotalPremium − TotalClaims) distribution is same for male and female.


In [53]:
# Severity among those with claims
severity_m = male[male["HasClaim"] == 1]["TotalClaims"]
severity_f = female[female["HasClaim"] == 1]["TotalClaims"]

t_stat, p_sev = stats.ttest_ind(severity_m, severity_f, equal_var=False)
print("Claim Severity difference (Male vs Female): t = {:.3f}, p = {:.5f}".format(t_stat, p_sev))

# Margin difference
margin_m = male["Margin"]
margin_f = female["Margin"]
t_m, p_m = stats.ttest_ind(margin_m, margin_f, equal_var=False)
print("Margin difference (Male vs Female): t = {:.3f}, p = {:.5f}".format(t_m, p_m))


Claim Severity difference (Male vs Female): t = -0.579, p = 0.56803
Margin difference (Male vs Female): t = -0.251, p = 0.80155


A/B Hypothesis Testing Results — Gender (Male vs Female)

Claim Frequency

Z = 0.201, p = 0.84049

Claim frequency: Male vs Female — no substantial difference (p ≫ 0.05)

→ No statistically significant difference in the proportion of policies with at least one claim between male and female policy‑holders.

Claim Severity (only among policies with claims)

t = –0.579, p = 0.56803

Mean claim amounts: not meaningfully different across genders (p ≫ 0.05)

→ No statistically significant difference in average claim amount between males and females who have claims.

Margin (TotalPremium − TotalClaims)

t = –0.251, p = 0.80155

Mean margin: no meaningful difference between genders (p ≫ 0.05)

→ No statistically significant difference in average profit margin per policy between male and female policy‑holders.

Interpretation & Business Implications (Gender)

Based on the tested KPIs (claim frequency, claim severity, and margin), there is no evidence that gender (male vs female) affects risk or profitability in a statistically meaningful way in your dataset.

In practical terms, this suggests that — under the current data and plan structure — gender should not be used as a segmentation criterion for risk‑based premium adjustments or targeting. Policies for men and women appear to perform similarly, so gender‑based differentiation might not yield risk‑adjusted benefits (at least from this analysis).

This supports a more neutral underwriting and pricing policy with respect to gender — which may also be desirable from fairness or regulatory perspectives.

# Province-Pair Balance Analysis: Find Two Most Balanced Provinces

This notebook computes covariate balance metrics for every pair of provinces in the dataset, based on selected control variables.  
We compute:

- For numeric variables: Standardized Mean Difference (SMD)  
- For categorical variables: difference in proportions for top categories  

Finally, we rank province pairs by overall balance (e.g. max absolute SMD across numeric covariates) and output the top k most balanced pairs for further outcome comparison (claims, margin, etc.).


In [54]:


numeric_covars = ["CustomValueEstimate", "TotalPremium", "SumInsured"]
categorical_covars = ["VehicleType", "CoverType", "Gender"]

provinces = df["Province"].dropna().unique().tolist()

def compute_smd(s1, s2):
    m1, m2 = s1.mean(), s2.mean()
    s1_std, s2_std = s1.std(), s2.std()
    pooled = np.sqrt((s1_std**2 + s2_std**2)/2)
    if pooled == 0:
        return np.nan
    return (m1 - m2)/pooled

def balance_for_pair(df, provA, provB):
    dfA = df[df["Province"] == provA]
    dfB = df[df["Province"] == provB]
    res = {"provA": provA, "provB": provB, "nA": len(dfA), "nB": len(dfB)}
    for col in numeric_covars:
        res[f"smd_{col}"] = compute_smd(dfA[col].dropna(), dfB[col].dropna())
    for col in categorical_covars:
        propA = dfA[col].value_counts(normalize=True)
        propB = dfB[col].value_counts(normalize=True)
        cats = set(propA.index) | set(propB.index)
        max_diff = max(abs(propA.get(c, 0) - propB.get(c, 0)) for c in cats)
        res[f"max_cat_diff_{col}"] = max_diff
    return res

records = []
for pa, pb in combinations(provinces, 2):
    rec = balance_for_pair(df, pa, pb)
    records.append(rec)

balance_df = pd.DataFrame(records)

balance_df["max_abs_smd"] = balance_df[[f"smd_{c}" for c in numeric_covars]].abs().max(axis=1)
balance_df["max_cat_diff"] = balance_df[[f"max_cat_diff_{c}" for c in categorical_covars]].max(axis=1)
balance_df["nA_plus_nB"] = balance_df["nA"] + balance_df["nB"]

balance_df = balance_df.sort_values(
    by=["max_abs_smd", "max_cat_diff", "nA_plus_nB"],
    ascending=[True, True, False]
).reset_index(drop=True)

print(balance_df.head(10))


          provA          provB      nA      nB  smd_CustomValueEstimate  \
0       Gauteng        Limpopo  393865   24836                 0.001616   
1       Gauteng     Free State  393865    8099                 0.024576   
2       Gauteng   Western Cape  393865  170796                -0.026879   
3    North West  Northern Cape  143287    6380                 0.018884   
4  Western Cape     North West  170796  143287                -0.031781   
5       Gauteng     Mpumalanga  393865   52718                 0.001808   
6    Mpumalanga     North West   52718  143287                -0.053827   
7    Mpumalanga        Limpopo   52718   24836                -0.001116   
8       Gauteng     North West  393865  143287                -0.041933   
9       Gauteng   Eastern Cape  393865   30336                -0.062827   

   smd_TotalPremium  smd_SumInsured  max_cat_diff_VehicleType  \
0         -0.005681        0.011496                  0.142710   
1         -0.022341        0.022145         

## Hypothesis Testing: Gauteng vs Limpopo

We compare two provinces — Gauteng and Limpopo — on the following KPIs:

- Claim Frequency (proportion of policies with at least one claim)  
- Claim Severity (average claim amount among policies with claims)  
- Margin (TotalPremium − TotalClaims) per policy  

Null hypotheses (for each KPI):

- H0_frequency: Claim frequency is equal in Gauteng and Limpopo.  
- H0_severity: Among policies with claims, average claim severity is equal in Gauteng and Limpopo.  
- H0_margin: Average margin is equal in Gauteng and Limpopo.

We use:

- Two-sample Z-test for proportions (frequency) — since we compare proportions across large samples. :contentReference[oaicite:2]{index=2}  
- Two-sample t-test (Welch’s, unequal variance) for claim severity and margin (since these are continuous numeric variables) — assuming sample size is large enough.  

We interpret p-values at α = 0.05 to decide whether to reject null hypotheses.  


In [55]:
# Derive KPI columns
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# Filter groups
prov_A = "Gauteng"
prov_B = "Limpopo"

dfA = df[df["Province"] == prov_A]
dfB = df[df["Province"] == prov_B]

print("Number of policies —", prov_A, ":", len(dfA), ",", prov_B, ":", len(dfB))

Number of policies — Gauteng : 393865 , Limpopo : 24836


### Claim Frequency — Two-sample Proportion Z-Test


In [56]:
from statsmodels.stats.proportion import proportions_ztest

count = np.array([ dfA["HasClaim"].sum(), dfB["HasClaim"].sum() ])
nobs  = np.array([ len(dfA), len(dfB) ])

z_stat, p_freq = proportions_ztest(count, nobs)
print(f"Claim Frequency comparison (Gauteng vs Limpopo): Z = {z_stat:.3f}, p = {p_freq:.5f}")

freq_A = dfA["HasClaim"].mean()
freq_B = dfB["HasClaim"].mean()
print(f"  -> {prov_A} frequency = {freq_A:.5f}, {prov_B} frequency = {freq_B:.5f}")


Claim Frequency comparison (Gauteng vs Limpopo): Z = 1.751, p = 0.07992
  -> Gauteng frequency = 0.00336, Limpopo frequency = 0.00270


### Claim Severity & Margin — Two-sample Welch t-tests (among policies with claims / all policies)


In [57]:
# Claim Severity (only for policies with a claim)
sevA = dfA[dfA["HasClaim"] == 1]["TotalClaims"]
sevB = dfB[dfB["HasClaim"] == 1]["TotalClaims"]

t_sev, p_sev = stats.ttest_ind(sevA, sevB, equal_var=False, nan_policy="omit")
print(f"Claim Severity (with claims) — {prov_A} vs {prov_B}: t = {t_sev:.3f}, p = {p_sev:.5f}")
print(f"  -> mean severity {prov_A}: {sevA.mean():.2f}, {prov_B}: {sevB.mean():.2f}")

# Margin (all policies)
marginA = dfA["Margin"]
marginB = dfB["Margin"]

t_m, p_m = stats.ttest_ind(marginA, marginB, equal_var=False, nan_policy="omit")
print(f"Margin difference — {prov_A} vs {prov_B}: t = {t_m:.3f}, p = {p_m:.5f}")
print(f"  -> mean margin {prov_A}: {marginA.mean():.2f}, {prov_B}: {marginB.mean():.2f}")


Claim Severity (with claims) — Gauteng vs Limpopo: t = 2.010, p = 0.04789
  -> mean severity Gauteng: 22243.88, Limpopo: 15171.29
Margin difference — Gauteng vs Limpopo: t = -3.132, p = 0.00174
  -> mean margin Gauteng: -13.56, Limpopo: 20.97


A/B Hypothesis Testing Results — Gauteng vs Limpopo
Results

Claim Frequency

Z-test statistic = 1.751, p-value = 0.0799

Observed frequencies: Gauteng = 0.00336, Limpopo = 0.00270

→ p-value > 0.05 → no statistically significant difference in claim frequency between the two provinces.

Claim Severity (among policies with claims)

Welch’s t-test: t = 2.010, p-value = 0.0479

Mean claim amount: Gauteng ≈ 22,243.88; Limpopo ≈ 15,171.29

→ p-value < 0.05 → there is a statistically significant difference: when claims occur, they tend to be more costly in Gauteng than in Limpopo.

Margin (TotalPremium − TotalClaims, per policy)

Welch’s t-test: t = –3.132, p-value = 0.00174

Mean margin: Gauteng ≈ –13.56, Limpopo ≈ 20.97

→ p-value ≪ 0.05 → significant difference in margin: on average, policies in Limpopo yield a positive margin, whereas in Gauteng the net margin is negative (or low) under current premium settings.

Interpretation & Business Implications

There is no strong evidence that the probability of having a claim at all (claim frequency) is different between Gauteng and Limpopo — the rate of claim occurrence appears roughly similar.

However, severity of claims (i.e. average claim size when a claim occurs) is significantly higher in Gauteng. This suggests that although claims are neither more nor less frequent, the financial impact per claim is substantially larger in Gauteng.

Because of higher claim severity combined with similar claim frequency, the overall profitability (margin) per policy is better in Limpopo: the data shows Limpopo has a positive average margin, while Gauteng shows a negative (or low) margin.

Business takeaway: Limpopo emerges as a relatively lower-risk / more profitable segment, making it a promising candidate for competitive pricing or increased marketing to attract clients. In contrast, Gauteng appears as a higher-risk, lower-margin region — suggesting a need for more conservative pricing, stricter underwriting, or risk-adjusted premiums when insuring clients there.

## Zip‑Code‑Based Segmentation: Identifying Balanced Zip‑Code Pair for Hypothesis Testing

Because the dataset contains many different postal codes (zip codes), we cannot directly compare all at once without risking confounding bias due to differences in other covariates (vehicle value, plan types, etc.).  
Therefore, we first **search across all pairs of zip‑codes** to identify a pair whose distributions of key covariates (vehicle value, premium, plan type, vehicle type, etc.) are as similar as possible ("balanced").  

We then treat that pair as a quasi‑“A/B” comparison (Group A = Zipcode A, Group B = Zipcode B), and test whether risk (claim frequency), claim severity, or margin differ — i.e. whether zip‑code itself appears to be associated with risk / profitability, after balancing observed covariates.

We use **Standardized Mean Difference (SMD)** for numeric variables and **max difference in proportions** for categorical variables as our balance metrics. We select the pair with lowest imbalance on all covariates, and with sufficient sample size to ensure statistical power.


In [58]:


# numeric summaries
num_means = df_sub.groupby("PostalCode")[numeric_covars].mean()
num_stds  = df_sub.groupby("PostalCode")[numeric_covars].std()
zip_counts = df_sub["PostalCode"].value_counts()

# categorical proportions (ensure all valid zips included)
cat_props = {}
for col in categorical_covars:
    props = (
        df_sub.groupby("PostalCode")[col]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
    )
    # reindex to include all valid zips, fill missing with 0
    props = props.reindex(valid_zips, fill_value=0)
    cat_props[col] = props

pairs = []
TOL = 0.2  # 20% tolerance

for zipA, zipB in combinations(valid_zips, 2):
    # only proceed if both zips have policies
    if zipA not in zip_counts.index or zipB not in zip_counts.index:
        continue
    nA, nB = zip_counts[zipA], zip_counts[zipB]
    if nA == 0 or nB == 0:
        continue  # skip if no policies exist

    # skip if sample sizes too different
    if abs(nA - nB) / max(nA, nB) > TOL:
        continue  

    # numeric SMDs
    numeric_smds = []
    for col in numeric_covars:
        if zipA in num_means.index and zipB in num_means.index:
            m1, m2 = num_means.loc[zipA, col], num_means.loc[zipB, col]
            s1, s2 = num_stds.loc[zipA, col], num_stds.loc[zipB, col]
            pooled = np.sqrt((s1**2 + s2**2) / 2)
            val = (m1 - m2) / pooled if pooled != 0 else np.nan
            numeric_smds.append(abs(val))

    # categorical diffs
    cat_diffs = []
    for col in categorical_covars:
        if zipA in cat_props[col].index and zipB in cat_props[col].index:
            diffs = abs(cat_props[col].loc[zipA] - cat_props[col].loc[zipB])
            cat_diffs.append(diffs.max())

    # overall balance score
    if numeric_smds or cat_diffs:
        balance_score = max(numeric_smds + cat_diffs)
        total_n = nA + nB
        pairs.append({
            "zipA": zipA,
            "zipB": zipB,
            "balance_score": balance_score,
            "n_total": total_n
        })

balance_df = pd.DataFrame(pairs).sort_values(
    by=["balance_score", "n_total"], ascending=[True, False]
).reset_index(drop=True)

print("Top 10 most balanced zip‑code pairs with similar sample sizes (only where policies exist):")
print(balance_df.head(10))


Top 10 most balanced zip‑code pairs with similar sample sizes (only where policies exist):
   zipA  zipB  balance_score  n_total
0   407  1852       0.032911     3778
1  3610  4137       0.033706    11928
2  1632  1804       0.039496     6182
3  4380  4031       0.043371     1364
4  5247  4200       0.048176     2892
5  4091  4023       0.058512     7157
6  7441  7140       0.059243     8381
7  4074  4170       0.059561     1201
8  2092   404       0.061496     2466
9   360  1057       0.061927     7463


## Hypothesis Testing – Zip‑code Comparison

We selected a pair of zip‑codes identified as highly balanced in covariates from our prior balancing analysis.  
We now test the following null hypotheses (two‑sample comparisons):

- **H₀ (Claim Frequency):** The proportion of policies with at least one claim (“HasClaim”) is the same in both zip‑codes.  
- **H₀ (Claim Severity):** Among policies with a claim, the claim amounts (TotalClaims) have the same mean between zip‑codes.  
- **H₀ (Margin):** The average margin (TotalPremium − TotalClaims) is the same between zip‑codes.

We use:

- Two‑proportion z‑test for Claim Frequency. :contentReference[oaicite:0]{index=0}  
- Welch’s t‑test (unequal variance) for Claim Severity and Margin. :contentReference[oaicite:1]{index=1}  

Significance threshold: α = 0.05  


In [59]:
df.columns = df.columns.str.strip()

# Ensure PostalCode is string for consistent filtering
df["PostalCode"] = df["PostalCode"].astype(str)

# Derived variables
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

zipA = "407"
zipB = "1852"

dfA = df[df["PostalCode"] == zipA]
dfB = df[df["PostalCode"] == zipB]

print("ZipA", zipA, "count:", len(dfA))
print("ZipB", zipB, "count:", len(dfB))

# Claim Frequency
count = np.array([dfA["HasClaim"].sum(), dfB["HasClaim"].sum()])
nobs  = np.array([len(dfA), len(dfB)])
z_stat, p_val = sm.stats.proportions_ztest(count, nobs)
print("Claim Frequency: Z = {:.3f}, p = {:.5f}".format(z_stat, p_val))
print("  → freq A:", dfA["HasClaim"].mean(), "; freq B:", dfB["HasClaim"].mean())

# Claim Severity (claims only)
sevA = dfA[dfA["HasClaim"] == 1]["TotalClaims"]
sevB = dfB[dfB["HasClaim"] == 1]["TotalClaims"]
if len(sevA) > 1 and len(sevB) > 1:
    t_stat_sev, p_sev = stats.ttest_ind(sevA, sevB, equal_var=False)
    print("Claim Severity: t = {:.3f}, p = {:.5f}".format(t_stat_sev, p_sev))
    print("  → mean severity A:", sevA.mean(), "; B:", sevB.mean())
else:
    print("Not enough claims in one group to compare severity")

# Margin comparison
marginA = dfA["Margin"]
marginB = dfB["Margin"]
t_stat_m, p_m = stats.ttest_ind(marginA, marginB, equal_var=False)
print("Margin diff: t = {:.3f}, p = {:.5f}".format(t_stat_m, p_m))
print("  → mean margin A:", marginA.mean(), "; B:", marginB.mean())


ZipA 407 count: 1975
ZipB 1852 count: 1803
Claim Frequency: Z = 1.226, p = 0.22021
  → freq A: 0.004556962025316456 ; freq B: 0.0022185246810870773
Claim Severity: t = -1.023, p = 0.37506
  → mean severity A: 10499.956140350876 ; B: 37603.45394736853
Margin diff: t = 0.507, p = 0.61202
  → mean margin A: 9.981702502178672 ; B: -26.024231792061624


Interpretation

Claim Frequency: The p‑value (≈ 0.22) is greater than a typical significance threshold (α = 0.05). Therefore, we fail to reject H₀₁: there is no statistically significant evidence of a difference in claim frequency between the two ZIP codes.

Claim Severity: With p ≈ 0.375, we fail to reject H₀₂: the difference in average claim severity (for those with claims) is not statistically significant under this test.

Margin: With p ≈ 0.612, we fail to reject H₀₃: there is no statistically significant difference in average margin between ZIP 407 and ZIP 1852.

In other words — based on this sample — we do not have evidence to claim that ZIP 407 and ZIP 1852 differ in claim frequency, severity, or profitability margin at the conventional 5% significance level.

What this means for risk‑segmentation

Because none of the tested metrics show a statistically significant difference between the two ZIP codes, the data does not support the hypothesis that ZIP 407 and ZIP 1852 represent different risk segments — at least not based on claim frequency, severity, or margin.

From a business / underwriting perspective: you would not treat these ZIP codes differently (e.g., by offering different premium pricing) based solely on the criteria tested here.